In [ ]:
import os

print(os.listdir('/content'))


['.config', 'sample_data']


In [ ]:
!pip install -q ultralytics opencv-python-headless numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.2 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import math
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm

try:
    from google.colab import files
    def upload_video():
        print("Upload a video file (MP4, AVI, etc.):")
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            print(f"Uploaded file: {filename}")
            return f"/content/{filename}"
        else:
            raise ValueError("No file uploaded!")
        return None
except ImportError:

    def upload_video():
        manual_path = input("Enter the video path: ").strip()
        if os.path.exists(manual_path):
            return manual_path
        else:
            raise ValueError("File not found!")


SOURCE_VIDEO = upload_video()
WEIGHTS = "yolov8n.pt"   # pretrained
CONF = 0.35
BASE_GREEN = 20
OUT_NAME = "/content/output_basic.mp4"

model = YOLO(WEIGHTS)
class_names = model.model.names

cap = cv2.VideoCapture(SOURCE_VIDEO)
if not cap.isOpened():
    raise SystemExit(f"Cannot open video: {SOURCE_VIDEO}")

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25
frame_area = w * h

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUT_NAME, fourcc, fps, (w, h))

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
pbar = tqdm(total=frame_count, desc="Processing (Basic)")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, conf=CONF, verbose=False)
    boxes = results[0].boxes

    vehicle_boxes = []
    total_area = 0
    emergency = False

    for b in boxes:
        xyxy = b.xyxy[0].cpu().numpy().astype(int)
        x1, y1, x2, y2 = xyxy
        cls_id = int(b.cls[0].cpu().numpy())
        name = class_names.get(cls_id, str(cls_id)).lower()

        if name in ["car", "truck", "bus", "motorcycle", "bicycle", "train"]:
            vehicle_boxes.append((x1, y1, x2, y2, name))
            total_area += max(0, x2 - x1) * max(0, y2 - y1)
        if "ambulance" in name or "fire" in name:
            emergency = True
            vehicle_boxes.append((x1, y1, x2, y2, name))
            total_area += max(0, x2 - x1) * max(0, y2 - y1)

    occupancy = (total_area / frame_area) * 100
    if occupancy < 5:
        congestion = "LOW"
    elif occupancy < 15:
        congestion = "MEDIUM"
    else:
        congestion = "HIGH"

    green_time = BASE_GREEN
    status = "NORMAL"
    if emergency:
        green_time = max(green_time, 30)
        status = "EMERGENCY PRIORITY"
    elif congestion == "LOW":
        green_time = max(5, green_time - 5)
        status = "LOW TRAFFIC"
    elif congestion == "MEDIUM":
        status = "MEDIUM TRAFFIC"
    else:
        green_time += 10
        status = "HIGH TRAFFIC"

    # Draw boxes
    for (x1, y1, x2, y2, name) in vehicle_boxes:
        color = (0, 255, 0)
        if "ambulance" in name or "fire" in name:
            color = (0, 0, 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, name, (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # Traffic light box (top-right)
    if emergency:
        tl = (0, 255, 0)
    elif congestion == "LOW":
        tl = (0, 0, 255)
    elif congestion == "MEDIUM":
        tl = (0, 255, 255)
    else:
        tl = (0, 255, 0)
    cv2.rectangle(frame, (w - 130, 30), (w - 70, 130), tl, -1)
    cv2.putText(frame, f"{int(green_time)}s", (w - 135, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicles: {len(vehicle_boxes)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(frame, f"Congestion: {congestion} ({occupancy:.1f}%)", (10, 64), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f"Signal Time: {int(green_time)}s", (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f"Status: {status}", (10, 136), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    if emergency:
        cv2.putText(frame, "EMERGENCY VEHICLE DETECTED!", (10, h - 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 3)

    out.write(frame)
    pbar.update(1)

pbar.close()
cap.release()
out.release()
print("Saved:", OUT_NAME)

Upload a video file (MP4, AVI, etc.):


Saving WhatsApp Video 2025-11-02 at 19.59.09_5a8b6981.mp4 to WhatsApp Video 2025-11-02 at 19.59.09_5a8b6981 (1).mp4
Uploaded file: WhatsApp Video 2025-11-02 at 19.59.09_5a8b6981 (1).mp4


Processing (Basic): 100%|█████████▉| 4509/4517 [13:20<00:01,  5.63it/s]

Saved: /content/output_basic.mp4


In [ ]:
from google.colab import files
files.download('/content/output_basic.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>